# 线性方程与最小二乘

学习目标：根据方程组的形状和秩选择求解方法，用残差与条件数判断结果，并完成简单的直线拟合。

前置知识：矩阵乘法、线性方程组、秩、范数、数值容差。

运行环境：Python 3.12、NumPy 2.5；本章使用实数 float64 数组。

环境准备：见 [环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

各示例使用本章给出的输入，后续代码复用首次导入的 np。

## 1 求解两元方程

两种零件的单价分别为 x₀、x₁ 元。购买 2 件第一种和 1 件第二种共花 8 元，购买两种各 1 件共花 5 元。方程为 2x₀ + x₁ = 8、x₀ + x₁ = 5，相减可得 x₀ = 3，再代入得 x₁ = 2。

把系数放进形状为 (2, 2) 的 a，每行对应一个方程，每列对应一个未知量；金额放进形状为 (2,) 的 b。np.linalg.solve(a, b) 返回形状为 (2,) 的解 x，使 a @ x 接近 b。

solve 用于方阵且满秩的系统，即方程系数的行或列线性无关。浮点计算仍有舍入误差。本例数据量级为个位数，以 1e-12 元作为代回金额的绝对容差，并设 rtol=0，只检查绝对误差；这不是所有业务数据的通用精度要求。

In [1]:
import numpy as np

a = np.array([[2.0, 1.0], [1.0, 1.0]])
b = np.array([8.0, 5.0])
x = np.linalg.solve(a, b)
residual = b - a @ x

print(x)  # [3. 2.]：两种零件的单价。
print(x.shape, x.dtype)  # (2,) float64
print(residual)  # 接近 [0. 0.]；残差按“观测值减代回值”定义。
print(np.allclose(a @ x, b, rtol=0, atol=1e-12))  # True

[3. 2.]
(2,) float64
[0. 0.]
True


## 2 超定系统与直线拟合

现有 4 次观测，要用直线 y ≈ slope × t + intercept 描述读数 y 随时间 t 的变化。只有斜率和截距两个未知量，却有 4 个方程，这是超定系统（overdetermined system）。观测不一定恰好落在一条直线上，可用 np.linalg.lstsq 求最小二乘解，使残差的平方和尽可能小。

一般记系数矩阵形状为 (m, n)：m 是方程数，n 是未知量数；本例 m=4、n=2。每行是一个时刻的系数 [t, 1]，解的顺序为 [slope, intercept]。时间单位为秒，读数使用任意单位。np.column_stack 把时间数组和全 1 数组作为两列组合。

lstsq 同时返回解、残差平方和、数值秩和奇异值。先完整接收这四项；rcond=None 使用与矩阵尺寸、浮点精度有关的默认秩阈值。

In [2]:
t = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 2.0, 4.0])
design = np.column_stack((t, np.ones(t.size)))
coef, squared_residuals, rank, singular_values = np.linalg.lstsq(
    design, y, rcond=None
)

print(design)  # 每行 [时间, 1]，形状为 (4, 2)。
print(coef)  # 约 [0.9 0.9]：斜率和截距。
print(squared_residuals)  # 约 [0.7]，是残差平方和，不是各点的残差。
print(rank)  # 2：两列在默认阈值下独立。
print(singular_values)  # 两个奇异值，均明显大于零。

[[0. 1.]
 [1. 1.]
 [2. 1.]
 [3. 1.]]
[0.9 0.9]
[0.7]
2
[4.10003045 1.09075677]


继续使用这组拟合结果，分别查看预测值与逐点残差。残差向量 r = y − design @ coef 的形状为 (4,)；np.linalg.norm(r) 默认计算向量的二范数，即各分量平方和的平方根。

对一维右端 y，且 m > n、矩阵满列秩时，lstsq 返回的残差平方和数组形状为 (1,)。若 m ≤ n，或秩小于 n，这一返回项为空；不能把空数组解释为“误差等于零”，应自行计算残差。

In [3]:
predicted = design @ coef
residual = y - predicted
squared_error = np.sum(residual ** 2)

print(predicted)  # 约 [0.9 1.8 2.7 3.6]
print(residual)  # 约 [0.1 0.2 -0.7 0.4]
print(np.linalg.norm(residual))  # 约 0.837，单位与读数相同。
print(squared_error)  # 约 0.7，单位为读数单位的平方。
# 两种计算得到同一个平方和；1e-12 为本例量级下的舍入余量。
print(np.allclose(squared_residuals, [squared_error], rtol=0, atol=1e-12))

[0.9 1.8 2.7 3.6]
[ 0.1  0.2 -0.7  0.4]
0.8366600265340756
0.7
True


非方阵不满足 solve 的输入条件。下面仍是 3 个方程、2 个未知量，即使右端恰好能精确拟合，也不能传给 solve；应按任务选用 lstsq。

In [4]:
rectangular = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])
b = np.array([1.0, 2.0, 3.0])

try:
    np.linalg.solve(rectangular, b)
except np.linalg.LinAlgError as error:
    print(type(error).__name__)  # LinAlgError：系数矩阵不是方阵。
else:
    raise AssertionError("预期非方阵不能由 solve 求解")

x, _, rank, _ = np.linalg.lstsq(rectangular, b, rcond=None)
print(x, rank)  # 约 [1. 1.]，秩为 2。
print(np.linalg.norm(b - rectangular @ x))  # 接近零；超定不等于必有非零残差。

LinAlgError
[1. 1.] 2
1.047382306668854e-15


## 3 奇异与欠定系统

方阵也可能没有唯一解。若一行是另一行的倍数，系数信息重复，矩阵不满秩，称为奇异矩阵。np.linalg.matrix_rank 给出数值秩；本例两个方程的系数只有一行独立。

这里第二个方程正好是第一个的两倍，有无穷多个解，solve 无法选出唯一解并抛出 LinAlgError。

In [5]:
a = np.array([[1.0, 1.0], [2.0, 2.0]])
b = np.array([2.0, 4.0])

print(np.linalg.matrix_rank(a))  # 1，小于方阵阶数 2。
try:
    np.linalg.solve(a, b)
except np.linalg.LinAlgError as error:
    print(type(error).__name__)  # LinAlgError：奇异矩阵。
else:
    raise AssertionError("预期奇异矩阵不能由 solve 求解")

1
LinAlgError


把第二个方程右端改为 5，两个方程相互矛盾，无法同时精确满足。lstsq 仍可最小化残差；如果有多个达到最小残差的解，它返回其中二范数最小的解。

下面特意保留全部返回值，观察“残差平方和返回项为空，但直接计算的残差不为零”的情况。

In [6]:
a = np.array([[1.0, 1.0], [2.0, 2.0]])
b = np.array([2.0, 5.0])
x, squared_residuals, rank, singular_values = np.linalg.lstsq(a, b, rcond=None)

print(x)  # 约 [1.2 1.2]：两分量之和为 2.4。
print(squared_residuals, squared_residuals.shape)  # [] (0,)
print(rank, singular_values)  # 秩为 1，第二个奇异值接近零。
print(b - a @ x)  # 约 [-0.4 0.2]，并非零残差。
print(np.sum((b - a @ x) ** 2))  # 约 0.2。

[1.2 1.2]
[] (0,)
1 [3.16227766e+00 1.57009246e-16]
[-0.4  0.2]
0.19999999999999998


方程数少于未知量数，即 m < n，称为欠定系统（underdetermined system）。下面 2 个方程约束 3 个未知量：x₀ + x₁ = 2、x₂ = 3，仍有多个精确解。

lstsq 选择二范数最小的那个解。这是额外的数学选择规则；若实际任务还要求非负、整数或满足其他约束，需要支持相应约束的方法，lstsq 的参数不提供这些约束。

In [7]:
a = np.array([[1.0, 1.0, 0.0], [0.0, 0.0, 1.0]])
b = np.array([2.0, 3.0])
x, squared_residuals, rank, _ = np.linalg.lstsq(a, b, rcond=None)
another = np.array([2.0, 0.0, 3.0])

print(x, x.shape)  # 约 [1. 1. 3.]，形状为 (3,)。
print(squared_residuals, rank)  # []，秩为 2；空返回项不表示没有求解。
print(np.linalg.norm(b - a @ x))  # 接近零。
print(np.linalg.norm(b - a @ another))  # 0.0，另一组解也满足方程。
print(np.linalg.norm(x), np.linalg.norm(another))  # 约 3.317 < 3.606。

[1. 1. 3.] (3,)
[] 2
4.440892098500626e-16
0.0
3.3166247903553994 3.605551275463989


## 4 数值秩与阈值

matrix_rank 通过奇异值判断秩：大于阈值的奇异值才计入。本章不展开奇异值分解，只利用这些数值的大小判断数值秩。默认绝对阈值为最大奇异值乘以 max(m, n)，再乘以该浮点类型的机器精度 eps。lstsq 的 rcond=None 则使用 eps × max(m, n) 作为相对截断比例。

默认阈值考虑浮点计算误差，不自动反映测量误差。下面是无量纲的对角系数矩阵；为了演示“把 1e-6 以下的方向视为不可分辨”这一假设，显式设置 matrix_rank 的 tol=1e-6。lstsq 的 rcond 则是相对于最大奇异值的比例，本例最大奇异值为 1，因此采用 rcond=1e-6 对应同一数量级的截断。阈值应由数据精度和任务决定，不能为获得想要的秩随意调整。

In [8]:
a = np.diag([1.0, 1e-8])
b = np.array([1.0, 1e-8])
default_x, _, default_rank, s = np.linalg.lstsq(a, b, rcond=None)
cut_x, _, cut_rank, _ = np.linalg.lstsq(a, b, rcond=1e-6)

print(s)  # [1.e+00 1.e-08]
print(np.linalg.matrix_rank(a), np.linalg.matrix_rank(a, tol=1e-6))  # 2 1
print(default_rank, default_x)  # 2，约 [1. 1.]。
print(cut_rank, cut_x)  # 1，约 [1. 0.]：小奇异值对应的方向被舍弃。
print(np.linalg.norm(b - a @ cut_x))  # 1e-8：截断改变了解与残差。

[1.e+00 1.e-08]
2 1
2 [1. 1.]
1 [1. 0.]
1e-08


## 5 病态输入与条件数

满秩只说明理论上存在唯一解，不保证解对输入误差不敏感。条件数很大的系统称为病态系统；np.linalg.cond 默认使用二范数条件数。对可逆方阵，它等于矩阵二范数与逆矩阵二范数的乘积，也等于最大奇异值与最小奇异值之比。

下面两行几乎相同。我们先用已知解 [1, 1] 构造右端，再仅把第二个右端增加 1e-10，比较解的变化。所有量均为无量纲合成数据；这次变化用于展示敏感性，不是模型中的随机噪声。

几何上，每个方程是一条直线，解是它们的交点。下图为看清线间距，把系数差 ε 取为 0.1：原方程为 x₀ + x₁ = 2、x₀ + (1 + ε)x₁ = 2 + ε；第二个右端再增加 ε 后，交点从 (1, 1) 移到 (0, 2)。图中的 ε 是本例参数，不是机器精度；下方代码取更小的 ε = 1e-10，线条在普通图上几乎重合，但解仍发生相同量级的变化。

![近乎平行的两条直线中第二条略微平移，交点从一一移动到零二的几何示意图](image/15-ill-conditioned-lines.png)

In [9]:
a = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-10]])
known_x = np.array([1.0, 1.0])
b = a @ known_x
changed_b = b + np.array([0.0, 1e-10])
x = np.linalg.solve(a, b)
changed_x = np.linalg.solve(a, changed_b)

print(np.linalg.matrix_rank(a))  # 2：默认阈值下仍是满秩。
print(np.linalg.cond(a))  # 约 4e10，条件数很大。
print(x, changed_x)  # 约 [1. 1.] 与 [0. 2.]，解明显改变。
print(np.linalg.norm(changed_b - b) / np.linalg.norm(b))  # 约 3.5e-11。
print(np.linalg.norm(changed_x - x) / np.linalg.norm(x))  # 约 1。

2
39999947702.450424
[1. 1.] [0. 2.]
3.535534198375736e-11
1.0


继续使用这组输入，检查 changed_x 的残差。它能很好地满足扰动后的方程，对原方程的残差也很小，却与原问题的已知解相差很大。

残差衡量代回方程后的差距，解误差衡量结果与目标解的差距。LAPACK 的误差分析把解误差界与条件数、经过尺度调整的残差联系起来；条件数很大时，残差小不能独自证明解可靠。实际任务应同时考虑输入误差、秩、条件数和残差，不以“未报错”或一次 allclose 为结论。

In [10]:
print(np.linalg.norm(changed_b - a @ changed_x))  # 接近零。
print(np.linalg.norm(b - a @ changed_x))  # 约 1e-10，原方程的残差仍很小。
print(np.linalg.norm(changed_x - known_x))  # 约 1.414，解误差明显。

# 以下 1e-9 只是刻意选定的代回绝对阈值，说明通过此检查仍不足以保证解准确。
print(np.allclose(a @ changed_x, b, rtol=0, atol=1e-9))  # True

0.0
1.000000082740371e-10
1.4142135623730951
True


## 6 选学：逆矩阵与伪逆

当任务需要逆矩阵本身时，用 np.linalg.inv；输入须为可逆方阵。求解 a @ x = b 则可直接用 solve，无须先把逆矩阵算出来。病态输入不会因改成 inv 就变得可靠，inv 也可能返回不准确的结果而不报错。

下面只对一个良态小矩阵比较两种表示，并用单位矩阵检查逆矩阵。此处复用 1e-12 的绝对容差作为小规模 float64 计算的舍入余量。

In [11]:
a = np.array([[2.0, 1.0], [1.0, 1.0]])
b = np.array([8.0, 5.0])
inverse = np.linalg.inv(a)

print(inverse)  # 约 [[1. -1.], [-1. 2.]]。
print(inverse @ b)  # 约 [3. 2.]。
print(np.allclose(a @ inverse, np.eye(2), rtol=0, atol=1e-12))  # True
print(np.allclose(inverse @ a, np.eye(2), rtol=0, atol=1e-12))  # True

[[ 1. -1.]
 [-1.  2.]]
[3. 2.]
True
True


np.linalg.pinv 计算 Moore–Penrose 伪逆，适用于矩形矩阵或秩亏矩阵。形状为 (m, n) 的输入得到形状为 (n, m) 的伪逆；与右端相乘可得到最小二乘解。它通过奇异值截断处理较小方向，并非把不存在的普通逆矩阵“修好”。

pinv 与 lstsq 的默认截断参数不同。下面对同一个秩亏问题显式使用相同的 rcond=1e-12：最大奇异值的万亿分之一作为相对截断尺度，远高于这个例子零奇异值的舍入量级，且远低于非零奇异值。这个阈值只服务于本例比较。

In [12]:
a = np.array([[1.0, 1.0], [2.0, 2.0]])
b = np.array([2.0, 5.0])
pseudo_inverse = np.linalg.pinv(a, rcond=1e-12)
x = pseudo_inverse @ b
least_squares_x, _, _, _ = np.linalg.lstsq(a, b, rcond=1e-12)

print(pseudo_inverse.shape)  # (2, 2)
print(x)  # 约 [1.2 1.2]。
print(np.allclose(x, least_squares_x, rtol=0, atol=1e-12))  # True
print(a @ pseudo_inverse)  # 不是单位矩阵，不能当作普通逆矩阵。

(2, 2)
[1.2 1.2]
True
[[0.2 0.4]
 [0.4 0.8]]


## 7 选学：行列式的数值表示

np.linalg.det 计算方阵的行列式。二阶矩阵 [[a, b], [c, d]] 的行列式为 ad − bc，其中 a、b、c、d 是四个矩阵元素。

行列式很大或很小时，直接计算可能溢出或下溢。np.linalg.slogdet 返回符号 sign 与绝对值的自然对数 logabsdet；实数矩阵的 sign 为 −1、0 或 1。保持对数表示可避免直接表示过大的乘积；重新指数化仍可能溢出。行列式为零时，返回 sign=0、logabsdet=−Inf。

下面对角矩阵的系数都很小，但条件数为 1。这个例子也说明，不能仅凭行列式数值很小就认定矩阵病态。

In [13]:
small = np.array([[2.0, 1.0], [1.0, 1.0]])
tiny = np.diag([1e-200, 1e-200])
singular = np.array([[1.0, 1.0], [2.0, 2.0]])

print(np.linalg.det(small))  # 1.0，手算为 2 × 1 - 1 × 1。
print(np.linalg.det(tiny))  # 0.0：真实乘积 1e-400 下溢，不能据此说矩阵奇异。
sign, logabsdet = np.linalg.slogdet(tiny)
print(sign, logabsdet)  # 1.0，约 -921.034：保留非零行列式的对数表示。
print(np.linalg.cond(tiny))  # 1.0
print(np.linalg.slogdet(singular))  # sign 为 0，logabsdet 为 -inf。

1.0
0.0
1.0 -921.0340371976183
1.0
SlogdetResult(sign=np.float64(0.0), logabsdet=np.float64(-inf))


## 本章小结

（1）方阵且满秩时用 solve 求唯一解；非方阵或秩亏系统可按最小二乘目标使用 lstsq，多个最优解中取二范数最小者。

（2）lstsq 返回解、残差平方和、秩和奇异值。残差平方和返回项为空时，要自己代回计算，不能当成零误差。

（3）数值秩依赖阈值。检查方程形状与秩后，还要结合输入精度、条件数和残差判断结果；残差小不保证病态问题的解可靠。

（4）inv、pinv 和 slogdet 分别服务于逆矩阵、伪逆与行列式对数表示，不能代替对问题条件的判断。

## 练习

（1）求解并代回

求解 3x₀ + x₁ = 7、x₀ + 2x₁ = 4。先手算，再用 solve，分别检查解的形状、系数矩阵的秩和残差。解释选用 1e-12 绝对容差的适用范围。

In [14]:
a = np.array([[3.0, 1.0], [1.0, 2.0]])
b = np.array([7.0, 4.0])

# 在此求解、打印并代回；检查结果形状为 (2,)，残差二范数不超过 1e-12。
# 写下手算过程和容差理由。

（2）预测返回项

下列系统有几个未知量？先预测解的形状、rank 和 squared_residuals 的形状，再运行核对。另写一个满足方程的不同解，比较两者二范数，说明 lstsq 采用的选择规则。

In [15]:
a = np.array([[1.0, 1.0, 1.0]])
b = np.array([3.0])
x, squared_residuals, rank, _ = np.linalg.lstsq(a, b, rcond=None)

print(x, x.shape)
print(rank, squared_residuals.shape)
# 在此给出另一组解；分别检查代回残差和解的二范数。

[1. 1. 1.] (3,)
1 (0,)


（3）改变观测条件

先根据前两个点拟合一条直线；再加入第三个不共线的观测点，仍要求找到残差平方和最小的直线。分别选择 solve 或 lstsq 并说明理由，包括矩阵形状与秩条件。是否可以因为第三个点加入后仍有两个未知量，就继续调用 solve？

In [16]:
t = np.array([0.0, 1.0, 2.0])
y = np.array([1.0, 3.0, 4.5])
design = np.column_stack((t, np.ones(t.size)))

# 在此分别处理 design[:2]、y[:2] 与全部观测，打印系数及残差。
# 检查第一组可精确代回；第二组给出最小二乘拟合，并解释方法选择。

（4）判断解是否可靠

右端第二个分量有 1e-7 的不确定变化。分别求解原始与变化后的系统，报告条件数、右端相对变化、解的相对变化和代回残差。若代回原方程的绝对容差设为 1e-6，检查通过能否证明解可靠？结合观察说明理由。

In [17]:
a = np.array([[1.0, 1.0], [1.0, 1.0 + 1e-7]])
known_x = np.array([1.0, 1.0])
b = a @ known_x
changed_b = b + np.array([0.0, 1e-7])

# 在此比较两次求解；相对变化使用二范数之比，分母取原始向量的二范数。
# 将数值观察和可靠性判断分开写，不只输出 allclose 的布尔结果。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（numpy.org） | NumPy 2.5，核查日期：2026-09-20。[solve](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.solve.html) 的 Parameters、Returns、Raises、Notes：方阵满秩、输入输出形状与失败条件；[lstsq](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.lstsq.html) 的定义、Parameters、Returns：最小二乘、最小范数、rcond 和空残差数组；[column_stack](https://numpy.org/doc/2.5/reference/generated/numpy.column_stack.html) 的定义、Examples：按列组合设计矩阵；[matrix_rank](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.matrix_rank.html) 的 tol、Notes：数值秩与误差阈值；[cond](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.cond.html) 的 Parameters、Notes 和 [inv](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.inv.html) 的 Notes、Examples：条件数、病态输入与奇异值比；[norm](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.norm.html) 的 Parameters、Notes：向量二范数；[allclose](https://numpy.org/doc/2.5/reference/generated/numpy.allclose.html) 的 Notes：容差公式；[pinv](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.pinv.html) 的 Parameters、Notes：伪逆、截断与最小二乘；[det](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.det.html) 的 Examples 和 [slogdet](https://numpy.org/doc/2.5/reference/generated/numpy.linalg.slogdet.html) 的定义、Returns：行列式及对数表示。 |
| LAPACK 官方文档（netlib.org） | LAPACK Users’ Guide，第三版：[Linear Least Squares (LLS) Problems](https://www.netlib.org/lapack/lug/node27.html)：超定、欠定与最小范数解；[Standard Error Analysis](https://www.netlib.org/lapack/lug/node78.html)：条件数、病态与误差放大（第 5 节几何示意的背景，2026-09-21 复核）；[Further Details: Error Bounds for Linear Equation Solving](https://www.netlib.org/lapack/lug/node81.html)：残差、尺度调整后的后向误差和条件数共同决定解误差界，支持“残差小不能独自证明解可靠”的边界。 |